# Data Structures (with `pandas`)

This notebook will teach you the basics of `pandas`, Python's go-to library for
working with tables (a.k.a. DataFrames).

In [7]:
import pandas as pd

URL = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2023/2023-10-17/'

## Declaring DataFrames

A DataFrame is just flat data (rows and columns) plus an *index* that labels
each row. Let's build the exact same table in three different ways.

### From a Dictionary

Each key is a column name, and each value is a list with that column's values.

In [8]:
# Each key-value pair is a column
df_dict = pd.DataFrame({'col1': [1, 2, 3], 'col2': [4, 5, 6]})
df_dict

,col1,col2
0,1,4
1,2,5
2,3,6


### From a List of Rows

Each element of the list is a row. Rows don't know the names of their columns,
so we have to pass them ourselves using `columns`.

In [9]:
# Each tuple is a row (lists work too!)
df_rows = pd.DataFrame(data=[(1, 4), (2, 5), (3, 6)], columns=['col1', 'col2'])
df_rows

,col1,col2
0,1,4
1,2,5
2,3,6


### From a List of Dictionaries

Each dictionary is a *record* (a row), and its keys tell pandas which column
each value belongs to. This is how a lot of data looks when it comes from the
web (e.g., JSON from an API).

In [10]:
records = [
    {'col1': 1, 'col2': 4},
    {'col1': 2, 'col2': 5},
    {'col1': 3, 'col2': 6},
]
df_records = pd.DataFrame(records)
df_records

,col1,col2
0,1,4
1,2,5
2,3,6


In [11]:
# Check that all three are truly the same table
print(df_dict.equals(df_rows))
print(df_dict.equals(df_records))

True
True


Notice the `0, 1, 2` on the left of every table? That's the **index**. pandas
adds one automatically if we don't give it one. The values are just flat data;
the index is what lets us find rows.

## Indexes and Slicing

Time for some real data! We'll use the Taylor Swift dataset from
[TidyTuesday](https://github.com/rfordatascience/tidytuesday/tree/main/data/2023/2023-10-17).
We can read it directly from a URL, so there's nothing to download.

In [12]:
# Read the albums table directly from the web
albums = pd.read_csv(URL + 'taylor_albums.csv')
display(albums.head())

,album_name,ep,album_release,metacritic_score,user_score
0,Taylor Swift,False,2006-10-24,67.0,8.5
1,The Taylor Swift Holiday Collection,True,2007-10-14,NaN,NaN
2,Beautiful Eyes,True,2008-07-15,NaN,NaN
3,Fearless,False,2008-11-11,73.0,8.4
4,Speak Now,False,2010-10-25,77.0,8.6


In [13]:
# The default index (0, 1, ..., 13) is not very informative. Let's use album
# names instead
albums_idx = albums.set_index('album_name')
display(albums_idx.head())

,ep,album_release,metacritic_score,user_score
album_name,,,,
Taylor Swift,False,2006-10-24,67.0,8.5
The Taylor Swift Holiday Collection,True,2007-10-14,NaN,NaN
Beautiful Eyes,True,2008-07-15,NaN,NaN
Fearless,False,2008-11-11,73.0,8.4
Speak Now,False,2010-10-25,77.0,8.6


### `.loc`

Use `.loc` to select by **label** (i.e., index values and column names).

In [14]:
# A single row
print(albums_idx.loc['folklore'])

# A single cell: [row, column]
print(albums_idx.loc['folklore', 'metacritic_score'])

ep                       False
album_release       2020-07-24
metacritic_score          88.0
user_score                 9.0
Name: folklore, dtype: object
88.0


In [15]:
# A slice of rows and a list of columns
albums_idx.loc['1989':'folklore', ['album_release', 'metacritic_score']]

,album_release,metacritic_score
album_name,,
1989,2014-10-27,76.0
reputation,2017-11-10,71.0
Lover,2019-08-23,79.0
folklore,2020-07-24,88.0


### `.iloc`

Use `.iloc` to select by **position** (0, 1, 2, ...), just like we did with
lists in the previous notebook.

In [16]:
# First row
print(albums_idx.iloc[0])

# Rows in positions 6, 7 and 8, and columns in positions 1 and 2
albums_idx.iloc[6:9, [1, 2]]

ep                       False
album_release       2006-10-24
metacritic_score          67.0
user_score                 8.5
Name: Taylor Swift, dtype: object


,album_release,metacritic_score
album_name,,
1989,2014-10-27,76.0
reputation,2017-11-10,71.0
Lover,2019-08-23,79.0


**Watch out!** `.loc` slices *include* the last label, but `.iloc` slices
*exclude* the last position.

In [17]:
# 1989 is in position 6 and folklore is in position 9
print(albums_idx.loc['1989':'folklore'].shape)  # 4 rows
print(albums_idx.iloc[6:9].shape)  # 3 rows! Position 9 is excluded

(4, 4)
(3, 4)


Our index is now made of album names, so there is no row labeled `0` anymore.
Let's see what happens if we try to grab it with `.loc`.

In [19]:
albums_idx.loc[0]

KeyError: 0

Try to select the `user_score` of every album from `Lover` to `Midnights` using
`.loc`. Then, do the same using `.iloc`.

In [20]:
# Your code here

In [21]:
print(albums_idx.loc['Lover':'Midnights', 'user_score'])
print(albums_idx.iloc[8:14, 3])

album_name
Lover                          8.4
folklore                       9.0
evermore                       8.9
Fearless (Taylor's Version)    8.9
Red (Taylor's Version)         9.0
Midnights                      8.3
Name: user_score, dtype: float64
album_name
Lover                          8.4
folklore                       9.0
evermore                       8.9
Fearless (Taylor's Version)    8.9
Red (Taylor's Version)         9.0
Midnights                      8.3
Name: user_score, dtype: float64


### Boolean Masks

A **mask** is a Series of `True`/`False` values (one per row). We can use a mask
to filter rows based on conditions. When combined with `.loc`, we can slice both
the matching rows and specific columns at the same time!

In [22]:
# Create a boolean mask: which albums have a metacritic score of 80 or more?
mask = albums_idx['metacritic_score'] >= 80
print(mask)

# Slice rows where mask is True, along with specific columns using .loc
albums_idx.loc[mask, ['user_score', 'metacritic_score']]

album_name
Taylor Swift                           False
The Taylor Swift Holiday Collection    False
Beautiful Eyes                         False
Fearless                               False
Speak Now                              False
Red                                    False
1989                                   False
reputation                             False
Lover                                  False
folklore                                True
evermore                                True
Fearless (Taylor's Version)             True
Red (Taylor's Version)                  True
Midnights                               True
Name: metacritic_score, dtype: bool


,user_score,metacritic_score
album_name,,
folklore,9.0,88.0
evermore,8.9,85.0
Fearless (Taylor's Version),8.9,82.0
Red (Taylor's Version),9.0,91.0
Midnights,8.3,85.0


## Creating and Updating Columns

To declare a new column, assign values to a column name that doesn't exist yet.

In [23]:
# A constant: every row gets the same value
albums_idx['artist'] = 'Taylor Swift'

# From other columns: user scores are 0-10, so put them on a 0-100 scale
albums_idx['user_score_100'] = albums_idx['user_score'] * 10
albums_idx

,ep,album_release,metacritic_score,user_score,artist,user_score_100
album_name,,,,,,
Taylor Swift,False,2006-10-24,67.0,8.5,Taylor Swift,85.0
The Taylor Swift Holiday Collection,True,2007-10-14,NaN,NaN,Taylor Swift,NaN
Beautiful Eyes,True,2008-07-15,NaN,NaN,Taylor Swift,NaN
Fearless,False,2008-11-11,73.0,8.4,Taylor Swift,84.0
Speak Now,False,2010-10-25,77.0,8.6,Taylor Swift,86.0
Red,False,2012-10-22,77.0,8.5,Taylor Swift,85.0
1989,False,2014-10-27,76.0,8.2,Taylor Swift,82.0
reputation,False,2017-11-10,71.0,8.3,Taylor Swift,83.0
Lover,False,2019-08-23,79.0,8.4,Taylor Swift,84.0


To update a column, assign values to a column name that already exists. We can
also update only the rows that meet a condition (remember `if` statements?) by
passing the condition to `.loc`.

In [24]:
# Overwrite `album_release` (text) with actual dates
albums_idx['album_release'] = pd.to_datetime(albums_idx['album_release'])

# Init a column with a default value, then update the rows that meet a condition
albums_idx['acclaimed'] = False
albums_idx.loc[albums_idx['metacritic_score'] >= 80, 'acclaimed'] = True
albums_idx[['album_release', 'metacritic_score', 'acclaimed']]

,album_release,metacritic_score,acclaimed
album_name,,,
Taylor Swift,2006-10-24,67.0,False
The Taylor Swift Holiday Collection,2007-10-14,NaN,False
Beautiful Eyes,2008-07-15,NaN,False
Fearless,2008-11-11,73.0,False
Speak Now,2010-10-25,77.0,False
Red,2012-10-22,77.0,False
1989,2014-10-27,76.0,False
reputation,2017-11-10,71.0,False
Lover,2019-08-23,79.0,False


Try to declare a column named `score_gap` that contains the difference between
`metacritic_score` and `user_score_100`.

In [25]:
# Your code here

In [26]:
albums_idx['score_gap'] = albums_idx['metacritic_score'] - albums_idx['user_score_100']
albums_idx[['metacritic_score', 'user_score_100', 'score_gap']]

,metacritic_score,user_score_100,score_gap
album_name,,,
Taylor Swift,67.0,85.0,-18.0
The Taylor Swift Holiday Collection,NaN,NaN,NaN
Beautiful Eyes,NaN,NaN,NaN
Fearless,73.0,84.0,-11.0
Speak Now,77.0,86.0,-9.0
Red,77.0,85.0,-8.0
1989,76.0,82.0,-6.0
reputation,71.0,83.0,-12.0
Lover,79.0,84.0,-5.0


## Joins

Joins combine two tables using a column they have in common (the *key*). Let's
load the songs table, which shares `album_name` with `albums`.

In [27]:
songs = pd.read_csv(URL + 'taylor_album_songs.csv')
songs.head()

,album_name,ep,album_release,track_number,track_name,artist,featuring,bonus_track,promotional_release,single_release,...,liveness,valence,tempo,time_signature,duration_ms,explicit,key_name,mode_name,key_mode,lyrics
0,Taylor Swift,False,2006-10-24,1,Tim McGraw,Taylor Swift,NaN,False,NaN,2006-06-19,...,0.1210,0.425,76.009,4.0,232107.0,False,C,major,C major,NaN
1,Taylor Swift,False,2006-10-24,2,Picture To Burn,Taylor Swift,NaN,False,NaN,2008-02-03,...,0.0962,0.821,105.586,4.0,173067.0,False,G,major,G major,NaN
2,Taylor Swift,False,2006-10-24,3,Teardrops On My Guitar,Taylor Swift,NaN,False,NaN,2007-02-19,...,0.1190,0.289,99.953,4.0,203040.0,False,A#,major,A# major,NaN
3,Taylor Swift,False,2006-10-24,4,A Place In This World,Taylor Swift,NaN,False,NaN,NaN,...,0.3200,0.428,115.028,4.0,199200.0,False,A,major,A major,NaN
4,Taylor Swift,False,2006-10-24,5,Cold As You,Taylor Swift,NaN,False,NaN,NaN,...,0.1230,0.261,175.558,4.0,239013.0,False,F,major,F major,NaN


In [28]:
# Keep things small: 7 albums (Fearless to folklore) and only their release date
albums_small = albums.iloc[3:10, [0, 2]]

# Only the first track of every album
first_tracks = songs.loc[songs['track_number'] == 1, ['album_name', 'track_name']]

print(albums_small)
print(first_tracks)

   album_name album_release
3    Fearless    2008-11-11
4   Speak Now    2010-10-25
5         Red    2012-10-22
6        1989    2014-10-27
7  reputation    2017-11-10
8       Lover    2019-08-23
9    folklore    2020-07-24
                      album_name                         track_name
0                   Taylor Swift                         Tim McGraw
15   Fearless (Taylor's Version)        Fearless (Taylor's Version)
41                     Speak Now                               Mine
58        Red (Taylor's Version)  State Of Grace (Taylor's Version)
88                          1989                Welcome To New York
104                   reputation                   ...Ready For It?
119                        Lover          I Forgot That You Existed
137                     folklore                              the 1
154                     evermore                             willow
171                    Midnights                      Lavender Haze


### Left Join

Keeps **every row of the left table** and brings in matching values from the
right table. If there is no match, we get `NaN`.

Why are `Fearless` and `Red` empty? The songs table only includes
*Taylor's Version* of re-released albums. We stan artists owning their songs!

In [29]:
pd.merge(albums_small, first_tracks, on='album_name', how='left')

,album_name,album_release,track_name
0,Fearless,2008-11-11,NaN
1,Speak Now,2010-10-25,Mine
2,Red,2012-10-22,NaN
3,1989,2014-10-27,Welcome To New York
4,reputation,2017-11-10,...Ready For It?
5,Lover,2019-08-23,I Forgot That You Existed
6,folklore,2020-07-24,the 1


In [33]:
cols_songs_info = ['track_name', 'track_number', 'album_name']
cols_songs_features = [
'track_name', 'bonus_track', 'tempo', 'liveness', 'valence'
]
songs_info = songs[cols_songs_info].copy()
songs_features = songs[cols_songs_features].copy()

In [35]:
songs_info.head()

,track_name,track_number,album_name
0,Tim McGraw,1,Taylor Swift
1,Picture To Burn,2,Taylor Swift
2,Teardrops On My Guitar,3,Taylor Swift
3,A Place In This World,4,Taylor Swift
4,Cold As You,5,Taylor Swift


In [38]:
pd.merge(left=albums, right=songs_info, on='album_name', how='left')

,album_name,ep,album_release,metacritic_score,user_score,track_name,track_number
0,Taylor Swift,False,2006-10-24,67.0,8.5,Tim McGraw,1.0
1,Taylor Swift,False,2006-10-24,67.0,8.5,Picture To Burn,2.0
2,Taylor Swift,False,2006-10-24,67.0,8.5,Teardrops On My Guitar,3.0
3,Taylor Swift,False,2006-10-24,67.0,8.5,A Place In This World,4.0
4,Taylor Swift,False,2006-10-24,67.0,8.5,Cold As You,5.0
...,...,...,...,...,...,...,...
193,Midnights,False,2022-10-21,85.0,8.3,"Would've, Could've, Should've",19.0
194,Midnights,False,2022-10-21,85.0,8.3,Dear Reader,20.0
195,Midnights,False,2022-10-21,85.0,8.3,Hits Different,14.0
196,Midnights,False,2022-10-21,85.0,8.3,"You're On Your Own, Kid (Strings Remix)",15.0


### Inner Join

Keeps **only the rows that match** in both tables.

In [30]:
pd.merge(albums_small, first_tracks, on='album_name', how='inner')

,album_name,album_release,track_name
0,Speak Now,2010-10-25,Mine
1,1989,2014-10-27,Welcome To New York
2,reputation,2017-11-10,...Ready For It?
3,Lover,2019-08-23,I Forgot That You Existed
4,folklore,2020-07-24,the 1


### Outer Join

Keeps **every row of both tables**, matched or not.

In [31]:
pd.merge(albums_small, first_tracks, on='album_name', how='outer')

,album_name,album_release,track_name
0,1989,2014-10-27,Welcome To New York
1,Fearless,2008-11-11,NaN
2,Fearless (Taylor's Version),NaN,Fearless (Taylor's Version)
3,Lover,2019-08-23,I Forgot That You Existed
4,Midnights,NaN,Lavender Haze
5,Red,2012-10-22,NaN
6,Red (Taylor's Version),NaN,State Of Grace (Taylor's Version)
7,Speak Now,2010-10-25,Mine
8,Taylor Swift,NaN,Tim McGraw
9,evermore,NaN,willow


To sum up:

| Join | Keeps | Rows here |
| :--- | :--- | :---: |
| `left` | All rows of the left table | 7 |
| `inner` | Only rows that match in both tables | 5 |
| `outer` | All rows of both tables | 12 |